# Fig. 3: Mean-field phase diagram in the $(N_c, N_g)$ plane

Linear stability of the mean-field solutions on a 1000 × 1000 grid of cooling atoms $N_c$ × gain atoms $N_g$, at $r = 0.5$. We count the unstable eigenvalues of

* the non-lasing solution of the full model (cavity, auxiliary mode $b$, cooling and gain atoms), and
* the lasing solution of the effective model (cooling atoms and mode $b$ adiabatically eliminated).

Together they give the phases of **panel (d)**.

**Outputs** (`../data/Nc_Ng/`): `mf_unstable_normal.npy`, `mf_unstable_lasing.npy`.

In [ ]:
using WignerSymbols
using LinearAlgebra
using NPZ

In [ ]:
Threads.nthreads()

In [ ]:
function dipole_me(F,Fp,m,p) # Dipole matrix element of the D1 line between |F,m> and |F',m+p> (Rb87, I = 3/2)
    
    J_g = 1/2
    J_e = 1/2
    I = 3/2

    sixj = Float64.(wigner6j(J_g, J_e, 1, Fp, F, I))
    cg = Float64.(clebschgordan(Fp, m+p, 1, -p, F, m))

    return (-1)^(Fp+J_g+1+I)*sqrt((2*Fp+1)*(2*J_g+1))*cg*sixj
end

## Parameters

In [ ]:
# Grid
num_Nc = 1000
num_Nl = 1000

# Rb87 level structure (B in G; angular frequencies, with the frequency in GHz)
B = 2.7
gamma = 2*pi*5.75*1e-3 # D1 linewidth
gamma_D2 = 2*pi*6.06*1e-3 # D2 linewidth
w_t = 2*pi*377107.463380
w_t_D2 = 2*pi*384230.48446
w_e = 2*pi*(6.834-3*0.7*1e-3*B)
w_s = w_t+2*pi*0.30544+2*pi*4.271677-2*pi*0.7*1e-3*B-2*pi*2*0.23*1e-3*B
w_r = w_t-2*pi*0.50906+2*pi*4.271677-2*pi*0.7*1e-3*B+2*pi*0.23*1e-3*B
w_u = w_t_D2-2*pi*0.22985+2*pi*4.271677-2*pi*0.7*1e-3*B-2*pi*0.93*1e-3*B

# Number of cooling (N_atom_I = N_c) and gain (N_atom_II = N_g) atoms
N_atom_I = logrange(100,2e4,num_Nc)
N_atom_II = logrange(10,2e4,num_Nl)
N_atom = N_atom_I .+ N_atom_II'

# Auxiliary cavity mode b
gb = 3*2*pi*1e-3
omega = 40*gamma_D2
w_b = w_u + 2*2pi
kappa_b = 20*2*pi*1e-3

# Main cavity mode a
kappa = 0.07*2*pi*1e-3
gr = 0.78*2*pi*1e-3
gs = gr*abs(dipole_me(2,2,-2,0)/dipole_me(1,1,-1,0))
r = 0.5 # Squeezing parameter
eps_cav = 1*1e-3*2*pi
w_cav = (1/3)*(w_e+4*w_r-w_s)+eps_cav

In [ ]:
function P_select(P_pk)

    P_cap = 200*1e-6*2pi

    if P_pk<P_cap
        return P_pk
    else
        return P_cap
    end
end

## Adiabatic elimination

Laser frequencies, rotating-frame detunings and effective couplings obtained after eliminating the excited levels $u$, $r$, $s$ and then the modes $b$ and $s$ (steps 1–7), and the effective pumping, decay and dephasing rates of the atoms (step 8).

In [ ]:
# Step 1: find convenient frequencies for the lasers

w_lr = (-w_cav+3*w_r-w_s).-sqrt.(4*gr^2*N_atom.+(w_cav-w_r)^2)
w_ls = -2*w_lr.+(4*w_r-w_s)
w_l = 0.5*(w_b.+w_lr.-w_ls.+w_u.+sqrt.(4*gb^2*N_atom_I.+(w_b-w_u)^2 .-omega^2))

# Step 2: move to rotating frame

delta_b = w_b .- w_l .- (w_ls.-w_lr)./2
delta_u = w_u .- w_l .- (w_ls.-w_lr)./2
delta_1 = w_e.-(w_ls.-w_lr)./2
delta_s = w_s.-w_ls
delta_r = w_r.-(w_ls.+w_lr)./2
delta_cav = w_cav.-(w_ls.+w_lr)./2

# Step 3: eliminate level u

w_0_b = -0.25.*(omega.^2)./delta_u
w_c_b = delta_b.-(N_atom_I*gb^2)./delta_u
chi_b = (gb^2)./delta_u
g = -omega.*gb./(2*delta_u)

# Step 4: find Rabi frequencies

omega_r = sqrt.((w_ls.-2*w_e.-w_lr)./(1 ./(w_lr.+w_ls.-2*w_r).-2*tanh.(r).^2 ./(w_ls.-w_s)))
omega_s = (gr/gs)*omega_r.*tanh.(r).*(delta_s./delta_r)

# Step 5: eliminate levels r and s

w_0 = delta_1.-0.25.*((omega_r.^2)./delta_r.-(omega_s.^2)./delta_s)
w_c = delta_cav.-(N_atom*gr^2)./delta_r
chi = (gr^2)./delta_r.-(gs^2)./delta_s
lambda_r = -omega_r.*gr./(2*delta_r)
lambda_s = -omega_s.*gs./(2*delta_s)
g_02 = (lambda_r.^2).-lambda_s.^2

alfa = omega_r./(2*delta_r)
beta = gr./delta_r
delta = omega_s./(2*delta_s)
xi = gs./delta_s
alfa_b = omega./(2*delta_u)
beta_b = gb./delta_u

dec_eff = gamma*dipole_me(1,1,-1,0)^2*alfa.^2
P_eff = gamma*dipole_me(2,2,-2,0)^2*delta.^2
dep_eff = (gamma*dipole_me(1,2,-1,-1)^2*delta.^2 .+ gamma*dipole_me(2,1,-2,+1)^2*alfa.^2)/4
dec_eff_b = gamma_D2*(5/12)*alfa_b.^2
dep_eff_b = gamma_D2*(1/10)*alfa_b.^2/4

# Step 6: Eliminate mode b

dec = (g.^2*kappa_b)./((w_c_b.-w_0_b).^2 .+ kappa_b^2/4)
delta_lamb = w_0_b .+ w_0 .+ N_atom_I.*(g.^2).*(w_c_b.-w_0_b)./((kappa_b^2/4).+(w_c_b.-w_0_b).^2) # coefficient of s's term

# Step 7: Eliminate mode s

dec_f = dec.*g_02./(dec.^2/4 .+ (delta_lamb./N_atom_I).^2)
delta_lamb_f = (g_02.*delta_lamb.*N_atom_I)./((delta_lamb.^2).+(N_atom_I.*dec/2).^2)

# Step 8: include pumping and dephasing

dec_T = dec_eff
dep_T = dep_eff

Γ_c = (4*g_02./(kappa.+dec_f)) # Purcell decay of qubits
P = P_select.((N_atom_II'.*Γ_c)/2 .-dec_T.-2*dep_T)

P_T = P.+P_eff

C = (2*N_atom_II'.*g_02)./(((P_T.+dec_T)/2 .+ 2*dep_T).*(dec_f.+kappa))./(1 .+(4*delta_lamb_f.^2)./((kappa.+dec_f).+(P_T.+dec_T.+4*dep_T)).^2)
n_q = (P_T.-dec_T)./(2*(kappa.+dec_f)).*N_atom_II'
control = P_T./(kappa.+dec_f).-((P_T.+dec_T)./(2*(kappa.+dec_f))).*(1 .+ 1 ./ C)

## Stability matrices

Jacobians of the mean-field equations around the non-lasing solution of the full model and around the lasing solution of the effective model.

In [ ]:
function mf_normal_full(κ,κb,g_0,g_b,Δ,Nc,Nl,P_c,γ_c,ϕ_c,P_l,γ_l,ϕ_l,χ)

    M = zeros(ComplexF64,4,4)
    
    sz_c = (P_c-γ_c)/(P_c+γ_c)
    sz_l = (P_l-γ_l)/(P_l+γ_l)
    Γ_l = P_l+γ_l+4*ϕ_l
    Γ_c = P_c+γ_c+4*ϕ_c
    M_c = P_c/(P_c+γ_c)
    M_l = P_l/(P_l+γ_l)

    M[1,1]=-κ/2
    M[1,3]=-im*g_0*Nc
    M[1,4]=-im*g_0*Nl

    M[2,2]=-κb/2-im*Δ-im*Nc*χ*M_c
    M[2,3]=-im*g_b*Nc

    M[3,1]=im*sz_c*g_0
    M[3,2]=im*sz_c*g_b
    M[3,3]=-(Γ_c/2+im*Δ)

    M[4,1]=im*g_0*sz_l
    M[4,4]=-Γ_l/2

    return M
end

In [ ]:
function mf_lasing_eff(Nl,κ,g0,ω_c,ω_0,P,dep,γ,Cop)
    M = zeros(ComplexF64,5,5)

    Ml = 1/2*(1+1/Cop)
    sz_l = 2*Ml - 1
    sz_n = (P-γ)/(P+γ)

    μ = -ω_c*(P+γ+4*dep)/(P+γ+4*dep+κ)

    Al = sqrt(Complex((Nl/κ)*(P-((P+γ)/2)*(1+1/Cop))))
    Sl = Al*(κ/2+im*(μ+ω_c))/(-im*g0*Nl)
    An = 0
    Sn = 0

    M[1,1]=-im*(ω_c+μ)-κ/2
    M[1,3]=-im*g0*Nl

    M[2,2]=im*(ω_c+μ)-κ/2
    M[2,4]=im*g0*Nl

    M[3,1]=im*g0*sz_l
    M[3,3]=-im*(ω_0+μ)-(P+γ)/2-2*dep
    M[3,5]=2*im*g0*Al

    M[4,2]=-im*g0*sz_l
    M[4,4]=im*(ω_0+μ)-(P+γ)/2-dep*2
    M[4,5]=-2*im*g0*conj(Al)

    M[5,1]=-im*g0*conj(Sl)
    M[5,2]=im*g0*Sl
    M[5,3]=im*g0*conj(Al)
    M[5,4]=-im*g0*Al
    M[5,5]=-(P+γ)

    return M
end

## Phase diagram

In [ ]:
eigs_comp = zeros(ComplexF64,num_Nc,num_Nl,4)

Threads.@threads for i in 1:num_Nc
    for j in 1:num_Nl
        eigs_comp[i,j,:] = eigvals(mf_normal_full(kappa,kappa_b,sqrt(g_02[i,j]),g[i,j],w_0_b[i,j],N_atom_I[i],N_atom_II[j],P_eff[i,j],dec_eff[i,j]+dec_eff_b[i,j],dep_eff[i,j]+dep_eff_b[i,j],P_T[i,j],dec_T[i,j],dep_T[i,j],chi_b[i,j]))
    end
end

PD_comp = sum((real(eigs_comp).>0);dims=3)[:,:,1] # Number of unstable eigenvalues of the non-lasing solution (full model)

eigs_eff_las = zeros(ComplexF64,num_Nc,num_Nl,5)

Threads.@threads for i in 1:num_Nc
    for j in 1:num_Nl
        eigs_eff_las[i,j,:] = eigvals(mf_lasing_eff(N_atom_II[j],(dec_f[i,j]+kappa),sqrt(g_02[i,j]),-delta_lamb_f[i,j],w_0[i,j],P[i,j]+P_eff[i,j],dep_eff[i,j],dec_eff[i,j],C[i,j]))
    end
end

PD_eff_las = sum((real(eigs_eff_las).>1e-9);dims=3)[:,:,1] # Number of unstable eigenvalues of the lasing solution (effective model)

## Save

In [ ]:
data_dir = joinpath("..", "data", "Nc_Ng") # Relative to this notebook

npzwrite(joinpath(data_dir, "mf_unstable_normal.npy"), PD_comp)
npzwrite(joinpath(data_dir, "mf_unstable_lasing.npy"), PD_eff_las)